# 🛡️ Giai Đoạn 1: Khảo Sát & Khám Phá Dữ Liệu NSL-KDD (EDA)
### Dự án: Network Intrusion & Cyber Attack Detection

Mục tiêu của giai đoạn này:
1. **Thiết lập môi trường** và nạp dữ liệu từ thư mục `D:\data_miner\archive`.
2. **Gán danh sách 41 đặc trưng + 2 cột nhãn** và phân thành 3 nhóm lớn (Basic, Content, Traffic/Host).
3. **Phân nhóm 5 lớp tấn công chuẩn**: `Normal`, `DoS`, `Probe`, `R2L`, `U2R`.
4. **Nhận diện 17 loại tấn công mới (Zero-Day)** chỉ có trong tập Test mà tập Train không có.
5. **Phát hiện hiện tượng lệch phân phối nặng (Heavy-tailed Skewness)** để chuẩn bị cho Giai đoạn 2 (Tiền xử lý).

In [ ]:
# [Bước 1] Import các thư viện cần thiết
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Cấu hình giao diện biểu đồ
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})
print("✅ Đã import thành công toàn bộ thư viện!")

--- 
## 1. Định Nghĩa Schema 41 Đặc Trưng & Đọc Dữ Liệu
File `KDDTrain+.txt` và `KDDTest+.txt` không có dòng tiêu đề (header). Chúng ta cần gán 43 cột theo đúng tài liệu chuẩn của NSL-KDD.

In [ ]:
# [Bước 2] Schema 43 cột phân thành 3 nhóm lý thuyết mạng
COLUMN_NAMES = [
    # --- Nhóm 1: Đặc trưng kết nối cơ bản (Basic features) ---
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent',
    
    # --- Nhóm 2: Đặc trưng nội dung gói tin (Content features) ---
    'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login',
    
    # --- Nhóm 3: Đặc trưng thống kê lưu lượng & máy chủ (Traffic/Host features) ---
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
    'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    
    # --- 2 Cột nhãn và độ khó ---
    'attack_type', 'difficulty_level'
]

DATA_DIR = Path(r"D:\data_miner\archive")
df_train = pd.read_csv(DATA_DIR / "KDDTrain+.txt", names=COLUMN_NAMES, header=None)
df_test = pd.read_csv(DATA_DIR / "KDDTest+.txt", names=COLUMN_NAMES, header=None)

print(f"📊 Kích thước tập Huấn luyện (KDDTrain+): {df_train.shape[0]:,} dòng x {df_train.shape[1]} cột")
print(f"📊 Kích thước tập Kiểm thử    (KDDTest+) : {df_test.shape[0]:,} dòng x {df_test.shape[1]} cột")
print(f"✅ Số lượng giá trị bị thiếu (Missing values): Train={df_train.isnull().sum().sum()}, Test={df_test.isnull().sum().sum()}")

In [ ]:
# [Bước 3] Xem trước 5 dòng đầu tiên của tập Train
preview_cols = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'attack_type', 'difficulty_level']
df_train[preview_cols].head()

--- 
## 2. Phân Loại 5 Nhóm Tấn Công Lớn (Normal, DoS, Probe, R2L, U2R)
NSL-KDD chứa hơn 30 loại tấn công con khác nhau. Ta quy về 5 lớp lớn để huấn luyện mô hình dễ dàng và chuẩn hóa.

In [ ]:
# [Bước 4] Ánh xạ từ điển 5 lớp tấn công
ATTACK_CATEGORIES = {
    'normal': 'Normal',
    # DoS (Denial of Service)
    'neptune': 'DoS', 'smurf': 'DoS', 'pod': 'DoS', 'teardrop': 'DoS',
    'land': 'DoS', 'back': 'DoS', 'apache2': 'DoS', 'udpstorm': 'DoS',
    'processtable': 'DoS', 'mailbomb': 'DoS',
    # Probe (Dò quét mạng)
    'portsweep': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe',
    'satan': 'Probe', 'saint': 'Probe', 'mscan': 'Probe',
    # R2L (Remote to Local)
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'phf': 'R2L',
    'multihop': 'R2L', 'warezmaster': 'R2L', 'warezclient': 'R2L', 'spy': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'snmpguess': 'R2L', 'snmpgetattack': 'R2L',
    'httptunnel': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'worm': 'R2L',
    # U2R (User to Root)
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R',
    'rootkit': 'U2R', 'ps': 'U2R', 'xterm': 'U2R', 'sqlattack': 'U2R'
}

df_train['attack_type'] = df_train['attack_type'].astype(str).str.strip().str.lower()
df_test['attack_type'] = df_test['attack_type'].astype(str).str.strip().str.lower()

df_train['attack_cat'] = df_train['attack_type'].map(lambda x: ATTACK_CATEGORIES.get(x, 'Unknown'))
df_test['attack_cat'] = df_test['attack_type'].map(lambda x: ATTACK_CATEGORIES.get(x, 'Unknown'))

# Bảng tổng hợp tỉ lệ 5 nhóm
train_dist = df_train['attack_cat'].value_counts()
test_dist = df_test['attack_cat'].value_counts()

df_summary = pd.DataFrame({
    'Train Count': train_dist,
    'Train %': (train_dist / len(df_train) * 100).round(2),
    'Test Count': test_dist,
    'Test %': (test_dist / len(df_test) * 100).round(2)
}).fillna(0)
df_summary

In [ ]:
# [Bước 5] Vẽ biểu đồ so sánh phân bố 5 lớp giữa Train và Test
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cats_order = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

sns.countplot(data=df_train, x='attack_cat', order=cats_order, ax=axes[0], palette='Blues_r')
axes[0].set_title("Phân bố trên Tập Train (KDDTrain+)", fontweight='bold')
axes[0].set_xlabel("Lớp lưu lượng")
axes[0].set_ylabel("Số lượng kết nối")

sns.countplot(data=df_test, x='attack_cat', order=cats_order, ax=axes[1], palette='Oranges_r')
axes[1].set_title("Phân bố trên Tập Test (KDDTest+)", fontweight='bold')
axes[1].set_xlabel("Lớp lưu lượng")
axes[1].set_ylabel("Số lượng kết nối")

plt.tight_layout()
plt.show()

--- 
## 3. Khám Phá Các Đòn Tấn Công Mới (Novel Attacks / Zero-Day)
Điểm đặc biệt nhất của tập `KDDTest+.txt` là nó chứa nhiều loại tấn công **chưa từng xuất hiện trong tập Train**. 

Hãy cùng tìm xem có bao nhiêu loại và bao nhiêu mẫu:

In [ ]:
# [Bước 6] Tìm các tấn công Zero-Day
train_attacks = set(df_train['attack_type'].unique())
test_attacks = set(df_test['attack_type'].unique())
novel_attacks = test_attacks - train_attacks

print(f"🔎 Có tổng cộng {len(novel_attacks)} loại tấn công mới toanh trong tập Test!")

novel_rows = []
for att in sorted(list(novel_attacks)):
    count = (df_test['attack_type'] == att).sum()
    cat = ATTACK_CATEGORIES.get(att, 'Unknown')
    novel_rows.append({'Tên Tấn Công Mới': att, 'Thuộc Nhóm': cat, 'Số Mẫu Trong Test': count})

df_novel = pd.DataFrame(novel_rows)
total_novel = df_novel['Số Mẫu Trong Test'].sum()
print(f"💥 Tổng số mẫu tấn công Zero-Day trong Test: {total_novel:,} mẫu ({total_novel / len(df_test):.2%})")
df_novel

--- 
## 4. Khảo Sát Phân Phối Lệch Nặng (Heavy-Tailed Skewness)
Các chỉ số như `src_bytes`, `dst_bytes`, `duration` có giá trị từ 0 lên tới hàng chục triệu byte. Nếu không nén bằng $\log(1 + x)$, mô hình sẽ bị hỏng!

In [ ]:
# [Bước 7] Bảng thống kê tả của các biến số lệch
skew_cols = ['duration', 'src_bytes', 'dst_bytes']
df_train[skew_cols].describe().round(1)

In [ ]:
# [Bước 8] So sánh phân phối trước và sau khi dùng log(1 + x)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(df_train['src_bytes'], ax=axes[0], color='red', fill=True, clip=(0, 50000))
axes[0].set_title("Phân phối src_bytes GỐC (Bị lệch cực nặng về 0)", fontweight='bold')
axes[0].set_xlabel("src_bytes (cắt ở 50,000)")

sns.kdeplot(np.log1p(df_train['src_bytes']), ax=axes[1], color='green', fill=True)
axes[1].set_title("Phân phối src_bytes SAU KHI DÙNG log(1 + x)", fontweight='bold')
axes[1].set_xlabel("log(1 + src_bytes)")

plt.tight_layout()
plt.show()

--- 
## 5. Khảo Sát Biến Phân Loại & Ma Trận Tương Quan (Correlation Heatmap)

In [ ]:
# [Bước 9] Kiểm tra các giá trị duy nhất của giao thức và dịch vụ mạng
print("Giao thức (protocol_type):", list(df_train['protocol_type'].unique()))
print("Cờ kết nối (flag):         ", list(df_train['flag'].unique()))

train_services = set(df_train['service'].unique())
test_services = set(df_test['service'].unique())
novel_services = test_services - train_services
print(f"Tổng số dịch vụ mạng (service) Train={len(train_services)}, Test={len(test_services)}")
if novel_services:
    print(f"⚠️ Có các dịch vụ mới trong Test: {novel_services}")
    print("   => Bắt buộc dùng OneHotEncoder(handle_unknown='ignore')!")

In [ ]:
# [Bước 10] Ma trận tương quan của các đặc trưng tỉ lệ lỗi mạng (Error Rates)
rate_cols = [
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 
    'same_srv_rate', 'diff_srv_rate', 'dst_host_serror_rate', 
    'dst_host_srv_serror_rate', 'dst_host_same_srv_rate'
]
corr_matrix = df_train[rate_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title("Ma trận tương quan các đặc trưng lưu lượng", fontweight='bold')
plt.show()

--- 
## 🎉 Kết Luận Giai Đoạn 1 & Bước Sang Giai Đoạn 2
Qua khảo sát EDA, chúng ta rút ra 4 kết luận then chốt:
1. **Dữ liệu rất sạch**, không bị missing values.
2. **Tập Test chứa 17 loại tấn công Zero-Day** (3,750 mẫu) mà Train không hề có $\rightarrow$ Chứng minh sự cần thiết của mô hình Unsupervised!
3. **`src_bytes` và `dst_bytes` bị lệch cực nặng** $\rightarrow$ Bắt buộc biến đổi $\log(1 + x)$ trước khi chuẩn hóa.
4. **Có dịch vụ lạ trong Test** $\rightarrow$ OneHotEncoder phải cấu hình `handle_unknown='ignore'`.